In [1]:
import os, cv2, json, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from diffusers import UNet3DConditionModel, DDPMScheduler
from IPython.display import Video, display

2026-01-22 18:09:05.826734: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769105346.012086      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769105346.065614      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769105346.483411      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769105346.483457      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769105346.483460      24 computation_placer.cc:177] computation placer alr

In [2]:
# ========== PATHS (UCF-101) ==========
DATA_ROOT = "/kaggle/input/eira-1-t2v-dataset"

VIDEO_ROOT = f"{DATA_ROOT}/UCF101/UCF-101"
SPLIT_ROOT = f"{DATA_ROOT}/UCF101TrainTestSplits-RecognitionTask/ucfTrainTestlist"

CLASS_LIST = f"{SPLIT_ROOT}/classInd.txt"
TRAIN_LIST = f"{SPLIT_ROOT}/trainlist01.txt"

SAVE_DIR = "/kaggle/working/t2v_quality"
os.makedirs(SAVE_DIR, exist_ok=True)


SAVE_DIR = "/kaggle/working/t2v_quality"
os.makedirs(SAVE_DIR, exist_ok=True)

# ========== VIDEO ==========
FRAME_SIZE = 96
FRAMES = 16
FPS = 6

# ========== MODEL ==========
LATENT_DIM = 8
TEXT_DIM = 512
MAX_LEN = 32

# ========== TRAINING ==========
VAE_EPOCHS = 3
DIFF_EPOCHS = 4
BATCH_SIZE = 1
LR = 1e-4

# ========== SAMPLING ==========
SAMPLE_STEPS = 40
GUIDANCE = 8.5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [3]:
# class → caption
class_map = {}
with open(CLASS_LIST) as f:
    for line in f:
        idx, name = line.strip().split()
        class_map[name] = f"a person {name.replace('_',' ').lower()}"

pairs = []
with open(TRAIN_LIST) as f:
    for line in f:
        path, _ = line.strip().split()
        cls = path.split("/")[0]
        caption = class_map[cls]
        pairs.append((path, caption))

pairs = pairs[:30000]   # realistic limit
print("Using videos:", len(pairs))

Using videos: 9537


In [4]:
special = ["<pad>","<bos>","<eos>","<unk>"]
word2idx = {t:i for i,t in enumerate(special)}

def tok(t): 
    return t.lower().split()

for _, cap in pairs:
    for w in tok(cap):
        if w not in word2idx:
            word2idx[w] = len(word2idx)

def encode(text):
    ids = [word2idx["<bos>"]]
    for w in tok(text):
        ids.append(word2idx.get(w, word2idx["<unk>"]))
        if len(ids) >= MAX_LEN-1:
            break
    ids.append(word2idx["<eos>"])
    while len(ids) < MAX_LEN:
        ids.append(word2idx["<pad>"])
    return np.array(ids, dtype=np.int64)

print("Vocab size:", len(word2idx))


Vocab size: 107


In [5]:
class UCF101Dataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def read_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []

        while len(frames) < FRAMES:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, (FRAME_SIZE, FRAME_SIZE))
            frame = torch.from_numpy(frame).permute(2,0,1).float() / 255.
            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            return torch.zeros(FRAMES, 3, FRAME_SIZE, FRAME_SIZE)

        while len(frames) < FRAMES:
            frames.append(frames[-1])

        return torch.stack(frames)

    def __getitem__(self, idx):
        rel_path, caption = self.pairs[idx]
        video = self.read_video(f"{VIDEO_ROOT}/{rel_path}")
        return {
            "video": video,
            "input_ids": torch.tensor(encode(caption)),
            "caption": caption
        }

    def __len__(self):
        return len(self.pairs)

train_loader = DataLoader(
    UCF101Dataset(pairs),
    batch_size=1,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


In [6]:
class VideoVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv3d(3, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv3d(64, 128, 4, 2, 1), nn.ReLU(),
            nn.Conv3d(128, LATENT_DIM, 3, 1, 1)
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose3d(LATENT_DIM, 128, 3, 1, 1), nn.ReLU(),
            nn.ConvTranspose3d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose3d(64, 3, 4, 2, 1), nn.Sigmoid()
        )

    def encode(self, x): return self.enc(x)
    def decode(self, z): return self.dec(z)
    def forward(self, x): return self.decode(self.encode(x))


class TextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(word2idx), TEXT_DIM)
        self.pos = nn.Embedding(MAX_LEN, TEXT_DIM)
        layer = nn.TransformerEncoderLayer(TEXT_DIM, 8, 2048, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, 6)

    def forward(self, ids):
        L = ids.size(1)
        pos = torch.arange(L, device=ids.device)
        return self.enc(self.emb(ids) + self.pos(pos))


vae = VideoVAE().to(DEVICE)
text_encoder = TextEncoder().to(DEVICE)

unet = UNet3DConditionModel(
    sample_size=(FRAMES//4, FRAME_SIZE//4, FRAME_SIZE//4),
    in_channels=LATENT_DIM,
    out_channels=LATENT_DIM,
    down_block_types=("CrossAttnDownBlock3D","CrossAttnDownBlock3D","DownBlock3D"),
    up_block_types=("UpBlock3D","CrossAttnUpBlock3D","CrossAttnUpBlock3D"),
    block_out_channels=(64,128,256),
    cross_attention_dim=TEXT_DIM
).to(DEVICE)

scheduler = DDPMScheduler(num_train_timesteps=500)


In [7]:
opt_vae = torch.optim.AdamW(vae.parameters(), lr=LR)
best_vae = 1e9

for e in range(1, VAE_EPOCHS+1):
    tot = 0
    for b in tqdm(train_loader, desc=f"VAE {e}/{VAE_EPOCHS}"):
        v = b["video"].to(DEVICE).permute(0,2,1,3,4)
        opt_vae.zero_grad()
        r = vae(v)
        min_t = min(r.shape[2], v.shape[2])
        loss = F.mse_loss(r[:,:, :min_t], v[:,:, :min_t])
        loss.backward()
        opt_vae.step()
        tot += loss.item()

    avg = tot / len(train_loader)
    print(f"[VAE] Epoch {e} loss:", avg)

    if avg < best_vae:
        best_vae = avg
        torch.save(vae.state_dict(), f"{SAVE_DIR}/best_vae.pt")
        print("⭐ Saved best VAE")

VAE 1/3:   0%|          | 0/9537 [00:00<?, ?it/s]

[VAE] Epoch 1 loss: 0.008845249804151615
⭐ Saved best VAE


VAE 2/3:   0%|          | 0/9537 [00:00<?, ?it/s]

[VAE] Epoch 2 loss: 0.004191871965561884
⭐ Saved best VAE


VAE 3/3:   0%|          | 0/9537 [00:00<?, ?it/s]

[VAE] Epoch 3 loss: 0.003205514025772913
⭐ Saved best VAE


In [8]:
opt = torch.optim.AdamW(
    list(unet.parameters()) + list(text_encoder.parameters()),
    lr=LR
)
best_diff = 1e9

for e in range(1, DIFF_EPOCHS+1):
    tot = 0
    for b in tqdm(train_loader, desc=f"DIFF {e}/{DIFF_EPOCHS}"):
        v = b["video"].to(DEVICE).permute(0,2,1,3,4)
        ids = b["input_ids"].to(DEVICE)

        with torch.no_grad():
            lat = vae.encode(v)

        t = torch.randint(0, scheduler.config.num_train_timesteps, (1,), device=DEVICE)
        noise = torch.randn_like(lat)
        noisy = scheduler.add_noise(lat, noise, t)

        emb = text_encoder(ids)
        pred = unet(noisy, t, encoder_hidden_states=emb).sample
        loss = F.mse_loss(pred, noise)

        opt.zero_grad()
        loss.backward()
        opt.step()
        tot += loss.item()

    avg = tot / len(train_loader)
    print(f"[DIFF] Epoch {e} loss:", avg)

    if avg < best_diff:
        best_diff = avg
        torch.save({
            "unet": unet.state_dict(),
            "text": text_encoder.state_dict()
        }, f"{SAVE_DIR}/best_diff.pt")
        print("⭐ Saved best diffusion")


DIFF 1/4:   0%|          | 0/9537 [00:00<?, ?it/s]

[DIFF] Epoch 1 loss: 0.16896781235801667
⭐ Saved best diffusion


DIFF 2/4:   0%|          | 0/9537 [00:00<?, ?it/s]

[DIFF] Epoch 2 loss: 0.12713893145216162
⭐ Saved best diffusion


DIFF 3/4:   0%|          | 0/9537 [00:00<?, ?it/s]

[DIFF] Epoch 3 loss: 0.11733094237603602
⭐ Saved best diffusion


DIFF 4/4:   0%|          | 0/9537 [00:00<?, ?it/s]

[DIFF] Epoch 4 loss: 0.110623547400755
⭐ Saved best diffusion


In [9]:
@torch.no_grad()
def generate(prompt):
    ids = torch.tensor(encode(prompt)).unsqueeze(0).to(DEVICE)
    txt = text_encoder(ids)
    null = text_encoder(torch.zeros_like(ids))

    lat = torch.randn(
        1, LATENT_DIM, FRAMES//4, FRAME_SIZE//4, FRAME_SIZE//4,
        device=DEVICE
    )

    scheduler.set_timesteps(SAMPLE_STEPS)
    for t in scheduler.timesteps:
        x = torch.cat([lat]*2)
        e = torch.cat([null, txt])
        p = unet(x, t, encoder_hidden_states=e).sample
        u, c = p.chunk(2)
        lat = scheduler.step(u + GUIDANCE*(c-u), t, lat).prev_sample

    return vae.decode(lat)

def save_video(t, path):
    v = t[0].permute(1,2,3,0).cpu().numpy()
    v = (v - v.min())/(v.max()-v.min()+1e-8)
    v = (v*255).astype("uint8")
    h,w = v.shape[1], v.shape[2]
    out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w,h))
    for f in v:
        out.write(f)
    out.release()
    print("Saved:", path)

video = generate("a person playing basketball")
out_path = f"{SAVE_DIR}/generated.mp4"
save_video(video, out_path)
display(Video(out_path))


Saved: /kaggle/working/t2v_quality/generated.mp4
